# OCI GenAI — Grok 3 Fast Quick StartDemonstrates how to call **xAI Grok 3 Fast** via Oracle Cloud Infrastructure (OCI) Generative AIusing the standard OpenAI Python SDK with API key authentication.**Requirements:**- `OCI_GENAI_API_KEY` environment variable (set via Codespaces secret)- `OCI_GENAI_ENDPOINT` environment variable (optional, defaults to Phoenix region)- `pip install openai`

In [ ]:
import osfrom openai import OpenAI# Load credentials from environment (set via Codespaces secrets)OCI_GENAI_ENDPOINT = os.environ.get(    "OCI_GENAI_ENDPOINT",    "https://inference.generativeai.us-phoenix-1.oci.oraclecloud.com/openai/v1")OCI_GENAI_API_KEY = os.environ["OCI_GENAI_API_KEY"]client = OpenAI(base_url=OCI_GENAI_ENDPOINT, api_key=OCI_GENAI_API_KEY)print(f"Client ready — endpoint: {OCI_GENAI_ENDPOINT}")

## 1. Basic Chat Completion

In [ ]:
response = client.chat.completions.create(    model="xai.grok-3-fast",    messages=[        {"role": "system", "content": "You are a helpful research assistant."},        {"role": "user", "content": "What is Oracle AI Vector Search? Explain in 3 sentences."}    ],    max_tokens=256,    temperature=0.7,)print(response.choices[0].message.content)print(f"\nTokens: {response.usage.total_tokens} (in={response.usage.prompt_tokens}, out={response.usage.completion_tokens})")

## 2. Streaming Response

In [ ]:
stream = client.chat.completions.create(    model="xai.grok-3-fast",    messages=[        {"role": "user", "content": "List 5 benefits of storing embeddings in Oracle Database."}    ],    max_tokens=512,    stream=True,)for chunk in stream:    delta = chunk.choices[0].delta    if delta.content:        print(delta.content, end="", flush=True)print()

## 3. Tool / Function Calling

In [ ]:
import jsontools = [    {        "type": "function",        "function": {            "name": "search_papers",            "description": "Search for academic papers on a given topic",            "parameters": {                "type": "object",                "properties": {                    "query": {"type": "string", "description": "Search query for papers"},                    "max_results": {"type": "integer", "description": "Maximum number of results"}                },                "required": ["query"]            }        }    }]response = client.chat.completions.create(    model="xai.grok-3-fast",    messages=[        {"role": "user", "content": "Find papers about memory architectures in AI agents"}    ],    tools=tools,    tool_choice="auto",)msg = response.choices[0].messageif msg.tool_calls:    for tc in msg.tool_calls:        print(f"Tool: {tc.function.name}")        print(f"Args: {json.dumps(json.loads(tc.function.arguments), indent=2)}")else:    print(f"Direct response: {msg.content}")

## 4. Multi-turn Conversation

In [ ]:
messages = [    {"role": "system", "content": "You are a research assistant specializing in AI and databases."},    {"role": "user", "content": "What is RAG?"}]# Turn 1r1 = client.chat.completions.create(model="xai.grok-3-fast", messages=messages, max_tokens=256)print("Turn 1:", r1.choices[0].message.content[:200])# Turn 2 — follow-upmessages.append({"role": "assistant", "content": r1.choices[0].message.content})messages.append({"role": "user", "content": "How does Oracle AI Vector Search fit into a RAG pipeline?"})r2 = client.chat.completions.create(model="xai.grok-3-fast", messages=messages, max_tokens=256)print("\nTurn 2:", r2.choices[0].message.content[:200])print(f"\nTotal tokens across 2 turns: {r1.usage.total_tokens + r2.usage.total_tokens}")

## 5. Retry Logic (handles 429 rate limits)When many workshop users hit the endpoint simultaneously, you may get 429 (rate limited) responses.This wrapper retries with exponential backoff.

In [ ]:
import timedef call_genai(messages, tools=None, model="xai.grok-3-fast",               max_retries=3, base_delay=2.0):    """Call OCI GenAI with automatic retry on 429 rate limits."""    kwargs = {"model": model, "messages": messages}    if tools:        kwargs["tools"] = tools        kwargs["tool_choice"] = "auto"    for attempt in range(max_retries + 1):        try:            return client.chat.completions.create(**kwargs)        except Exception as e:            status_code = getattr(e, 'status_code', None) or 0            if status_code == 429 and attempt < max_retries:                delay = base_delay * (2 ** attempt)                print(f"Rate limited. Retrying in {delay:.0f}s (attempt {attempt+1}/{max_retries})...")                time.sleep(delay)            else:                raise# Test itresp = call_genai([{"role": "user", "content": "Say hello!"}])print(resp.choices[0].message.content)

## Done!This notebook confirms OCI GenAI + Grok 3 Fast works with API key auth.The workshop notebook (`notebook_complete_ocigenai.ipynb`) uses the same setupfor the full Research Paper Assistant with Oracle AI Vector Search and memory engineering.